# Random Forest

Il Random Forest è un algoritmo di machine learning supervisionato basato su una collezionei alberi decisionali indipendenti, combinati per migliorare la accuratezza e la robustezza rispetto a singoli alberi.

In [ ]:
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from pathlib import Path
import warnings
# Nascondo i warning
warnings.filterwarnings('ignore')

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Lavoro su singola fold

In [ ]:
def fit_single_fold(train_idx, test_idx, features, target, groups, n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features):
  # ========== DEBUGGING: Stampo indici train/test  ==========

  """print("?"*50 + "\nDebug\n" + "?"*50)
  print(f"\nFold {fold} - File: {csv_name}")
  print(f"  Train indice: {train_index[:10]})")
  print(f"  Test indice: {test_index[:10]})")
  print(f"  Train gruppo (Patient IDs): {groups.iloc[train_index].unique()}")
  print(f"  Test gruppo (Patient IDs): {groups.iloc[test_index].unique()}")
  print("?"*100)"""
  # ==========================================================

  # Suddivido i dati in set di training e di test per la fold corrente
  X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
  y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

  rf = RandomForestClassifier(
      n_estimators= n_estimators,             
      max_depth= max_depth ,                
      min_samples_split= min_samples_split,            
      min_samples_leaf= min_samples_leaf,             
      max_features= max_features,  
  )

  # Addestro il modello sul train set di questa fold
  rf.fit(X_train, y_train)

  # Predico il target sul test set di questa fold
  y_pred = rf.predict(X_test)
  
  # DEBUG
  #score = f1_score(y_test, y_pred, average="micro")
  #print(f"Fold {fold} - {max_iter=}, {max_depth=}, {min_samples_leaf=}, Score={score:.4f}")

  return f1_score(y_test, y_pred, average="micro")

# Training

In [ ]:
def training(file_path, csv_name):

    # Legge il dataset
    df = pd.read_csv(file_path)

    # Definisco le colonne target
    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]']
    
    # Vado a rimuovere le lesioni (righe) non valide
    df_validi = df.dropna(subset=original_target_list).copy()

    # Trasformo tutto in valori binari per "facilitare" il lavoro
    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)  
    
    # Lista finale delle colonne target binarizzate che verranno usate per l'addestramento
    final_target_list = ['PR_class', 'ER_class', 'KI67_class']
    
    """
     Preparo le feature (X) e i target (y) per il modello
    """
    # Definisco tutte le colonne da rimuovere per ottenere solo le feature radiomiche
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    # 'target' contiene le 3 colonne da usare
    target = df_validi[final_target_list]
    
    # 'groups' contiene l'ID del paziente per ogni lesione.
    # Mi serve per fare la cross-validation a gruppo
    groups = df_validi['Patient ID']

    # Riempip a Nan se è rimasto vuoto
    features = features.fillna(features.mean())
    
    # Imposto la strategia di cross-validation.
    # GroupKFold assicura che le lesioni dello stesso paziente non vengano mai divise tra training set e test set
    cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Lista vuota per collezionare i punteggi di performance di ogni fold.
    scores = []

    # Definisco gli iperparametri
    iperparametri = {
      'n_estimators': [50],              # Numero di alberi nella foresta
      'max_depth': [None, 10],           # Profondità massima degli alberi
      'min_samples_split': [5],          # Campioni minimi per split
      'min_samples_leaf': [1],           # Campioni minimi in una foglia
      'max_features': ['log2'],          # Numero di feature per split
    }   

    for n_estimators in iperparametri['n_estimators']:
      for max_depth in iperparametri['max_depth']:
          for min_samples_split in iperparametri['min_samples_split']:
              for min_samples_leaf in iperparametri['min_samples_leaf']:
                  for max_features in iperparametri['max_features']:
                    fold_scores = Parallel(n_jobs=-1)(delayed(fit_single_fold)(train_idx, test_idx, features, target, groups, n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features) for train_idx, test_idx in cv.split(features, target, groups))

                    # calcolo media e deviazione standard degli score su tutte le fold
                    mean_score = np.mean(fold_scores)
                    std_score = np.std(fold_scores)

                    # registro i risultati per la combinazione di parametri corrente
                    scores.append({
                        'n_estimators': n_estimators,
                        'max_depth': max_depth,
                        'min_samples_leaf': min_samples_leaf,
                        'min_samples_split': min_samples_split,
                        'max_features': max_features,
                        'mean_score': mean_score,
                        'std_score': std_score,
                        'fold_scores': fold_scores
                    })
                  

    return scores

# Lettura dei file

In [ ]:
#TODO: GIA MODIFICATO


results_per_dataset = {}

for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Stampo i risultati per ogni dataset
for name, metrics_list in results_per_dataset.items():

    # Stampo solo i migliori
    best_result = max(metrics_list, key=lambda x: x['mean_score'])
    print(f"\nDataset: {name}:")
    print(f"  n_estimators: {best_result['n_estimators']}")
    print(f"  max_depth: {best_result['max_depth']}")
    print(f"  min_samples_leaf: {best_result['min_samples_leaf']}")
    print(f"  min_samples_split: {best_result['min_samples_split']}")
    print(f"  Mean F1-score: {best_result['mean_score']:.3f} ± {best_result['std_score']:.3f}\n")




# va dentro al for sopra


# LI ESEGUO DU COLAB, questo è l'output dato


Dataset: t2_medsam:
  n_estimators: 50
  max_depth: None
  min_samples_leaf: 2
  min_samples_split: 5
  Mean F1-score: 0.788 ± 0.083


Dataset: t2_preprocessed:
  n_estimators: 50
  max_depth: None
  min_samples_leaf: 2
  min_samples_split: 5
  Mean F1-score: 0.775 ± 0.060


Dataset: t2_original:
  n_estimators: 50
  max_depth: 10
  min_samples_leaf: 1
  min_samples_split: 5
  Mean F1-score: 0.776 ± 0.072


Dataset: medsam_dynamic:
  n_estimators: 50
  max_depth: 10
  min_samples_leaf: 2
  min_samples_split: 2
  Mean F1-score: 0.755 ± 0.080


Dataset: preprocessed_dynamic:
  n_estimators: 50
  max_depth: None
  min_samples_leaf: 1
  min_samples_split: 5
  Mean F1-score: 0.770 ± 0.031


Dataset: original_dynamic:
  n_estimators: 50
  max_depth: 10
  min_samples_leaf: 2
  min_samples_split: 2
  Mean F1-score: 0.760 ± 0.054


"""
print(f"\nNome CSV: {name}")
for res in metrics_list:
    scores_per_fold = res['fold_scores']
    formatted_scores = [f'{s:.3f}' for s in scores_per_fold]
    print(f"Params: n_estimators={res['n_estimators']}, max_depth={res['max_depth']}, "
            f"min_samples_split={res['min_samples_split']}, min_samples_leaf={res['min_samples_leaf']}")
    #print(f"Scores per fold: {formatted_scores}")
    print(f"Media e Dev. Std.: {res['mean_score']:.3f} ± {res['std_score']:.3f}\n")
"""